# Chapter 6 &mdash; Merging Equivalence Classes into the Minimal Machine

**Concept 9 of the Chapter 6 decomposition:** *Merging Equivalence Classes into the Minimal Machine*

Pairs still at $-1$ are equivalent; overlapping pairs coalesce into classes, one class per minimal state.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Merging-Equivalence-Classes/Concept-Merging-Equivalence-Classes.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The frame table leaves a set of equivalent **pairs**. Those pairs **overlap**: if
$(p,q)$ and $(q,r)$ are both equivalent, then $p$, $q$ and $r$ all belong to one
class &mdash; indistinguishability is transitive.

**Bashing** the pairs together (Jove's `bash_eql_classes`) coalesces them into
equivalence classes. Each class becomes **one state** of the minimal DFA; a
representative is chosen (`mk_rep_eqc`), transitions are lifted class-wise, and the
class containing $q_0$ is the new start state.

Only reachable states may be merged this way, which is why pruning comes first.

## 2. Definitions

### A machine with a three-way merge

In [ ]:
D = md2mc('''DFA
I  : 0 -> A
I  : 1 -> B
A  : 0 -> F1
A  : 1 -> F2
B  : 0 -> F2
B  : 1 -> F3
F1 : 0 | 1 -> F1
F2 : 0 | 1 -> F2
F3 : 0 | 1 -> F3
''')

### Equivalence classes, by transitive closure of the equivalent pairs

In [ ]:
from itertools import product
def classes(D):
    qs = sorted(D["Q"])
    def lang(q, n):
        return frozenset(''.join(p) for k in range(n+1)
                         for p in product(sorted(D["Sigma"]), repeat=k)
                         if run_dfa_h(D, ''.join(p), q) in D["F"])
    n = len(qs)
    groups = {}
    for q in qs:
        groups.setdefault(lang(q, n), []).append(q)
    return sorted(map(sorted, groups.values()))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;8.&nbsp;DFA Minimization by Frames: $k$-Distinguishability](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Frames-K-Distinguishability/Concept-Frames-K-Distinguishability.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;10.&nbsp;Minimization as a Fixed-Point Computation: `fixptDist`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Fixed-Point-Minimization/Concept-Fixed-Point-Minimization.ipynb)&nbsp;&rarr;

---

## 3. Tests

Three final states behave identically, so they form one class.

In [ ]:
cls = classes(D)
for c in cls: print("  class :", c)
print("\n%d states -> %d classes" % (len(D["Q"]), len(cls)))
assert any(len(c) >= 3 for c in cls), "F1, F2, F3 should coalesce"

Jove's `bash_eql_classes` performs the same coalescing on overlapping pairs.

In [ ]:
# bash_eql_classes takes a LIST OF PAIRS (tuples), not a list of sets,
# and returns [(representative, [members...]), ...] -- one entry per class.
demo = [('F1','F2'), ('F2','F3'), ('A','B')]
print("overlapping pairs :", demo)
for rep, members in bash_eql_classes(demo):
    print("   representative %-4s class %s" % (rep, sorted(members)))
print("\n(F1,F2) and (F2,F3) share F2, so they coalesce into one class of three.")
big = [sorted(m) for _, m in bash_eql_classes(demo) if len(m) >= 3]
assert big and set(big[0]) == {'F1','F2','F3'}

One class becomes one state, and the language is preserved.

In [ ]:
m = min_dfa(D)
print("minimal states :", sorted(m["Q"]))
print("count matches class count?", len(m["Q"]) == len(cls))
assert len(m["Q"]) == len(cls)
assert langeq_dfa(D, m)

The class containing $q_0$ is the new start state.

In [ ]:
print("original q0 : %s      minimal q0 : %s" % (D["q0"], m["q0"]))
print("minimal q0 name mentions the class members:", m["q0"])
assert accepts_dfa(m, '00') == accepts_dfa(D, '00')

## 4. Animation

The minimized machine &mdash; each state is a whole class of the original.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(D), FuseEdges=True)

## 5. Exercises


1. Why must indistinguishability be transitive for "bashing" to be well defined?
2. What goes wrong if you merge classes **before** pruning unreachable states?
3. Try `min_dfa(D, state_name_mode='verbose')`. What do the names show?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6/Concept-Merging-Equivalence-Classes')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')